In [1]:
import shared as shar
from pyspark import StorageLevel
import os

In [2]:
spark = shar.get_spark()

22/11/09 20:13:56 WARN Utils: Your hostname, pop-os resolves to a loopback address: 127.0.1.1; using 192.168.0.178 instead (on interface enp4s0)
22/11/09 20:13:56 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
:: loading settings :: url = jar:file:/home/vergenter/Sources/masterthesis/igraphvenv/lib/python3.10/site-packages/pyspark/jars/ivy-2.5.0.jar!/org/apache/ivy/core/settings/ivysettings.xml


Ivy Default Cache set to: /home/vergenter/.ivy2/cache
The jars for the packages stored in: /home/vergenter/.ivy2/jars
com.johnsnowlabs.nlp#spark-nlp_2.12 added as a dependency
org.neo4j#neo4j-connector-apache-spark_2.12 added as a dependency
:: resolving dependencies :: org.apache.spark#spark-submit-parent-810396d7-dd64-4b63-a1a9-17736fd0444e;1.0
	confs: [default]
	found com.johnsnowlabs.nlp#spark-nlp_2.12;4.0.0 in central
	found com.typesafe#config;1.4.2 in central
	found org.rocksdb#rocksdbjni;6.29.5 in central
	found com.amazonaws#aws-java-sdk-bundle;1.11.828 in central
	found com.github.universal-automata#liblevenshtein;3.0.0 in central
	found com.google.code.findbugs#annotations;3.0.1 in central
	found net.jcip#jcip-annotations;1.0 in central
	found com.google.code.findbugs#jsr305;3.0.1 in central
	found com.google.protobuf#protobuf-java-util;3.0.0-beta-3 in central
	found com.google.protobuf#protobuf-java;3.0.0-beta-3 in central
	found com.google.code.gson#gson;2.3 in central
	fo

22/11/09 20:13:57 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).


In [3]:
os.environ["NEO4J_LOGIN"]

'neo4j'

In [ ]:

articles_abstracts = spark.read.format("org.neo4j.spark.DataSource")\
  .option("url", "bolt://192.168.0.178:7687")\
  .option("authentication.basic.username", os.environ["NEO4J_LOGIN"])\
  .option("authentication.basic.password", os.environ["NEO4J_PASSWORD"]).option("query", "MATCH (n:Article) where n.language =\"en\" WITH n RETURN n.id as id,n.title +'. '+ n.abstract as text")\
  .option("partitions", "4")\
  .load()
articles_abstracts.persist(StorageLevel.DISK_ONLY)

In [ ]:
from sparknlp.base import DocumentAssembler,Pipeline,LightPipeline
from sparknlp.annotator import StopWordsCleaner,SentenceDetector,Tokenizer,YakeKeywordExtraction,BertForTokenClassification,NerConverter,LemmatizerModel,Normalizer,Word2VecModel
# Transforms the raw text into a document readable by the later stages of the
# pipeline
document_assembler = DocumentAssembler() \
    .setInputCol('text') \
    .setOutputCol('document')

# Separates the document into sentences
sentence_detector = SentenceDetector() \
    .setInputCols(['document']) \
    .setOutputCol('sentences')# \
    #.setDetectLists(True)

# Separates sentences into individial tokens (words)
tokenizer = Tokenizer() \
    .setInputCols(['sentences']) \
    .setOutputCol('tokens') \
    .setContextChars(['(', ')', '?', '!', '.', ','])

# The keyphrase extraction model. Change MinNGrams and MaxNGrams to set the
# minimum and maximum length of possible keyphrases, and change NKeywords to
# set the amount of potential keyphrases identified per document.
keywords = YakeKeywordExtraction() \
    .setInputCols('tokens') \
    .setOutputCol('keywords') \
    .setMinNGrams(2) \
    .setMaxNGrams(5) \
    .setNKeywords(10) \
    .setStopWords(StopWordsCleaner().getStopWords())


# Assemble all of these stages into a pipeline, then fit the pipeline on an
# empty data frame so it can be used to transform new inputs.
pipeline_keywords = Pipeline(stages=[
    document_assembler, 
    sentence_detector,
    tokenizer,
    keywords
])
empty_df = spark.createDataFrame([[""]]).toDF('text')
keywords_model = pipeline_keywords.fit(empty_df)

In [ ]:
with_keywords = keywords_model.transform(articles_abstracts).select(F.col("id"),F.arrays_zip(F.expr("transform(keywords.metadata,i -> i['score'])"),F.col("keywords.result")).alias("keywords"))
with_keywords.cache()

In [ ]:
query1 ="""UNWIND event.keywords as keyword
MERGE (k:Keyword{word:keyword.`1`})"""

query2 ="""match (a:Article{id:event.id})
UNWIND event.keywords as keyword
match (k:Keyword{word:keyword.`1`})
with a,k,toInteger(keyword.`0`) as score
MERGE (a)-[rel:ABOUT]->(k)
ON CREATE SET rel.score = score
ON MATCH SET rel.score = CASE WHEN rel.score > score THEN rel.score ELSE score END"""



with_keywords.write.format("org.neo4j.spark.DataSource")\
.option("url", "bolt://192.168.0.178:7687")\
.option("authentication.basic.username", os.environ["NEO4J_LOGIN"])\
.option("authentication.basic.password", os.environ["NEO4J_PASSWORD"])\
.option("transaction.retries", 5)\
.option("transaction.retry.timeout", 5)\
.mode("Overwrite")\
.option("query", query1)\
.save()

with_keywords.write.format("org.neo4j.spark.DataSource")\
.option("url", "bolt://192.168.0.178:7687")\
.option("authentication.basic.username", os.environ["NEO4J_LOGIN"])\
.option("authentication.basic.password", os.environ["NEO4J_PASSWORD"])\
.option("transaction.retries", 5)\
.option("transaction.retry.timeout", 5)\
.mode("Append")\
.option("query", query2)\
.save()

In [ ]:
from graphdatascience import GraphDataScience

# Use Neo4j URI and credentials according to your setup
gds = GraphDataScience("bolt://192.168.0.178:7687", auth=(os.environ["NEO4J_LOGIN"], os.environ["NEO4J_PASSWORD"]))

print(gds.version())
G = gds.graph.get(name)

In [ ]:
r2 = gds.pageRank

In [1]:
# tf is 1(is keyword in document) or 0(no keyword in document) over 10(keywords count for document)
# tf -to slowo i dokument
# idf słowo
# keyword idf = log(articles.lenght/keyword.degree)
# it answers how revelant is world for document
# if many document have this world it means that is not relevant
# bigger page rank means that document is more relevant
# keyword with highest degree will have most relevancy
# idf higher degree it means less

# tf will be normalized page rank
# firest idea tf-idf*page rank
#

1. get data
2. generate keyword
3. meause for keyword